In [ ]:
import pandas as pd
import os
import numpy as np
import librosa
from sklearn.metrics import precision_recall_fscore_support, confusion_matrix
AUDIO_DIR = "/kaggle/input/audio-files"   # your audio dataset
CSV_PATH = "/kaggle/input/silero-timestamps/timestamps_summary.csv"
SAMPLING_RATE = 16000
FRAME_MS = 30

CLIP_HALF = 1.0  # seconds

def frame_to_time(idx):
    return idx * FRAME_MS / 1000


In [ ]:
timestamps_df = pd.read_csv(CSV_PATH)
timestamps_df.head()


In [ ]:
def segments_to_frames(segments, audio_duration, frame_ms=30):
    n_frames = int((audio_duration * 1000) / frame_ms)
    labels = np.zeros(n_frames, dtype=int)

    for start, end in segments:
        s = int(start * 1000 / frame_ms)
        e = int(end * 1000 / frame_ms)
        labels[s:e] = 1

    return labels


In [ ]:
speaker_id = timestamps_df["speaker_id"].iloc[0]
audio_path = os.path.join(AUDIO_DIR, speaker_id)

y, sr = librosa.load(audio_path, sr=SAMPLING_RATE)
duration = len(y) / sr

segments = timestamps_df[timestamps_df["speaker_id"] == speaker_id][["start_s", "end_s"]].values
silero_frames = segments_to_frames(segments, duration)

print("Audio duration (s):", duration)
print("Silero speech ratio:", silero_frames.mean())


In [ ]:
!pip install webrtcvad


In [ ]:
import webrtcvad

def webrtc_vad_frames(y, sr, mode, frame_ms=30):
    vad = webrtcvad.Vad(mode)
    frame_len = int(sr * frame_ms / 1000)
    frames = librosa.util.frame(y, frame_length=frame_len, hop_length=frame_len).T

    labels = []
    for f in frames:
        pcm = (f * 32768).astype(np.int16).tobytes()
        labels.append(vad.is_speech(pcm, sr))

    return np.array(labels, dtype=int)



how strict or how sensitive the speech detection is.

In [ ]:
wrtc2 = webrtc_vad_frames(y, sr, mode=2)
wrtc3 = webrtc_vad_frames(y, sr, mode=3)

print("WebRTC-2 ratio:", wrtc2.mean())
print("WebRTC-3 ratio:", wrtc3.mean())


In [ ]:
def energy_vad_frames(y, sr, frame_ms=30, percentile=90):
    frame_len = int(sr * frame_ms / 1000)
    frames = librosa.util.frame(y, frame_length=frame_len, hop_length=frame_len).T
    energy = np.mean(frames**2, axis=1)

    thr = np.percentile(energy, percentile)
    return (energy > thr).astype(int)


In [ ]:
energy = energy_vad_frames(y, sr)
print("Energy VAD ratio:", energy.mean())


In [ ]:
energy_raw = np.mean(
    librosa.util.frame(y, frame_length=int(sr*0.03), hop_length=int(sr*0.03)).T**2,
    axis=1
)

print("Energy min:", energy_raw.min())
print("Energy max:", energy_raw.max())
print("Energy median:", np.median(energy_raw))


In [ ]:
min_len = min(len(silero_frames), len(wrtc2), len(wrtc3), len(energy))

silero_frames = silero_frames[:min_len]
wrtc2 = wrtc2[:min_len]
wrtc3 = wrtc3[:min_len]
energy = energy[:min_len]


In [ ]:
stack = np.vstack([silero_frames, wrtc2, wrtc3, energy])
consensus = (np.sum(stack, axis=0) >= 2).astype(int)

print("Consensus speech ratio:", consensus.mean())


In [ ]:
p, r, f1, _ = precision_recall_fscore_support(
    consensus,
    silero_frames,
    average="binary"
)

cm = confusion_matrix(consensus, silero_frames)

print("Precision:", round(p, 3))
print("Recall:", round(r, 3))
print("F1:", round(f1, 3))
print("\nConfusion Matrix:\n", cm)
